# Direct-count SAA comparison demo

This executable notebook reuses the six-type, 12-household fixture from `count_table_refactor_demo.ipynb`. It compares `count_native_ilp` with `count_native` and `legacy_exact` without assuming that their joint distributions should match. The direct ILP is sampling-free within each pass; its optional loop-back retention remains stochastic.

In [2]:
import random
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from PopSynthesis.Methods.IPSF.const import count_field, zone_field
from PopSynthesis.Methods.IPSF.SAA.operations.count_tables import counts_to_marginals, population_size
from PopSynthesis.Methods.IPSF.SAA.operations.wrapper_saa_run import saa_run
from PopSynthesis.Methods.IPSF.utils.condensed import condense_df

ATTRIBUTES = ['hhsize', 'vehicles', 'income']
TYPE_KEYS = ATTRIBUTES


## Six candidate types

The pool contains 10 weighted records across six unique household types. Pool frequency is a distributional reference, not a capacity limit.

In [3]:
pool_display = pd.DataFrame([
    ('A', '1',  '0',  'Low',  3),
    ('B', '1',  '1',  'High', 1),
    ('C', '2',  '1',  'Low',  2),
    ('D', '2',  '2+', 'High', 2),
    ('E', '3+', '1',  'Low',  1),
    ('F', '3+', '2+', 'High', 1),
], columns=['type_id', *ATTRIBUTES, 'pool_frequency'])
pool_count = pl.from_pandas(
    pool_display.drop(columns='type_id').rename(columns={'pool_frequency': count_field})
).with_columns(pl.col(count_field).cast(pl.Int64))
assert population_size(pool_count) == 10
pool_display


,type_id,hhsize,vehicles,income,pool_frequency
0,A,1,0,Low,3
1,B,1,1,High,1
2,C,2,1,Low,2
3,D,2,2+,High,2
4,E,3+,1,Low,1
5,F,3+,2+,High,1


## One-zone Census targets

Every controlled marginal sums to 12.

In [4]:
marginal_columns = pd.MultiIndex.from_tuples([
    ('hhsize', '1'), ('hhsize', '2'), ('hhsize', '3+'),
    ('vehicles', '0'), ('vehicles', '1'), ('vehicles', '2+'),
    ('income', 'Low'), ('income', 'High'),
])
target = pd.DataFrame(
    [[4, 5, 3, 3, 5, 4, 7, 5]],
    index=pd.Index([101], name=zone_field),
    columns=marginal_columns,
)
assert set(target.T.groupby(level=0).sum().iloc[:, 0]) == {12}
target


hhsize       vehicles       income     
             1  2 3+        0  1 2+    Low High
zone_id                                        
101          4  5  3        3  5  4      7    5

## Hand-check the first dense reference table

After direct initialization, the `hhsize` row totals are `(4, 5, 3)`. The table below calculates the exact pool-derived references for the `vehicles` adjustment.

In [5]:
vehicle_pool = pd.DataFrame(
    [[3, 1, 0], [0, 2, 2], [0, 1, 1]],
    index=['1', '2', '3+'],
    columns=['0', '1', '2+'],
)
row_totals = pd.Series([4, 5, 3], index=vehicle_pool.index)
vehicle_q = vehicle_pool.div(vehicle_pool.sum(axis=1), axis=0).mul(row_totals, axis=0)
expected_q = pd.DataFrame(
    [[3.0, 1.0, 0.0], [0.0, 2.5, 2.5], [0.0, 1.5, 1.5]],
    index=vehicle_pool.index, columns=vehicle_pool.columns,
)
pd.testing.assert_frame_equal(vehicle_q, expected_q)
vehicle_q


,0,1,2+
1,3.0,1.0,0.0
2,0.0,2.5,2.5
3+,0.0,1.5,1.5


## Run all three modes

The existing modes use fixed Python/NumPy seeds for repeatability. The direct mode also fixes CBC ordering and seeds. Only contractual totals, marginals, and schemas are asserted.

In [6]:
def run_mode(mode, *, seed=17, max_run_time=1, extra_rm_frac=0):
    random.seed(seed)
    np.random.seed(seed)
    rows, targets = saa_run(
        target.copy(),
        pool_count.clone(),
        considered_atts=ATTRIBUTES,
        ordered_to_adjust_atts=ATTRIBUTES,
        last_adjustment_order=ATTRIBUTES,
        max_run_time=max_run_time,
        extra_rm_frac=extra_rm_frac,
        stochastic_mode=mode,
    )
    return rows, condense_df(rows).sort([zone_field, *TYPE_KEYS]), targets

modes = ['count_native', 'legacy_exact', 'count_native_ilp']
runs = {mode: run_mode(mode) for mode in modes}
for mode, (rows, counts, targets) in runs.items():
    assert len(rows) == 12
    assert population_size(counts) == 12
    assert rows.columns == [*ATTRIBUTES, zone_field]
    assert targets == [12]
    observed = counts_to_marginals(counts, ATTRIBUTES)
    expected = target.copy(); expected.index = expected.index.astype(str)
    observed = observed.reindex(index=expected.index, columns=expected.columns, fill_value=0)
    pd.testing.assert_frame_equal(observed.astype(int), expected.astype(int), check_names=False)
print('All modes preserve the public schema, total population, and controlled marginals.')


ADJUSTING FOR hhsize
ADJUSTING FOR vehicles
DOING zone 101
ADJUSTING FOR income
DOING zone 101
All modes preserve the public schema, total population, and controlled marginals.


## Compare compressed joint distributions

Differences here are expected: `count_native_ilp` is a new algorithm rather than a representation-only refactor.

In [7]:
joint_tables = []
for mode, (_, counts, _) in runs.items():
    table = counts.to_pandas().drop(columns=zone_field).rename(columns={count_field: mode})
    joint_tables.append(table.set_index(TYPE_KEYS))
joint_comparison = pd.concat(joint_tables, axis=1).fillna(0).astype(int).sort_index()
joint_comparison


count_native  legacy_exact  count_native_ilp
hhsize vehicles income                                              
1      0        Low                3             3                 3
       1        High               1             1                 1
2      1        Low                3             1                 2
       2+       High               2             4                 3
3+     1        Low                1             3                 2
       2+       High               2             0                 1

In [8]:
pool_joint = pool_count.to_pandas().set_index(TYPE_KEYS)[count_field].astype(float)
all_keys = pool_joint.index.union(joint_comparison.index)
pool_probability = pool_joint.reindex(all_keys, fill_value=0) / pool_joint.sum()
distance_rows = []
for mode in modes:
    generated = joint_comparison[mode].reindex(all_keys, fill_value=0).astype(float)
    generated_probability = generated / generated.sum()
    total_variation = 0.5 * (generated_probability - pool_probability).abs().sum()
    distance_rows.append({'mode': mode, 'total_variation_from_pool': total_variation})
pool_distance = pd.DataFrame(distance_rows)
assert pool_distance['total_variation_from_pool'].between(0, 1).all()
pool_distance


,mode,total_variation_from_pool
0,count_native,0.116667
1,legacy_exact,0.283333
2,count_native_ilp,0.116667


## Exercise stochastic loop-backs in the new mode

Attribute adjustment remains direct-count ILP. The retained subset between passes is stochastic and uses the existing count-level removal/pruning mechanism.

In [9]:
loop_rows_1, loop_counts_1, loop_targets_1 = run_mode(
    'count_native_ilp', seed=29, max_run_time=3, extra_rm_frac=0.35
)
loop_rows_2, loop_counts_2, loop_targets_2 = run_mode(
    'count_native_ilp', seed=29, max_run_time=3, extra_rm_frac=0.35
)
assert loop_targets_1 == loop_targets_2
assert loop_counts_1.equals(loop_counts_2)
assert len(loop_rows_1) == 12
observed = counts_to_marginals(loop_counts_1, ATTRIBUTES)
expected = target.copy(); expected.index = expected.index.astype(str)
observed = observed.reindex(index=expected.index, columns=expected.columns, fill_value=0)
pd.testing.assert_frame_equal(observed.astype(int), expected.astype(int), check_names=False)
print('Residual pass sizes:', loop_targets_1)
loop_counts_1


Residual pass sizes: [12, 4, 1]


hhsize,vehicles,income,zone_id,count
str,str,str,str,u32
"""1""","""0""","""Low""","""101""",3
"""1""","""1""","""High""","""101""",1
"""2""","""1""","""Low""","""101""",2
"""2""","""2+""","""High""","""101""",3
"""3+""","""1""","""Low""","""101""",2
"""3+""","""2+""","""High""","""101""",1


## Inspect compact direct-mode diagnostics

In [10]:
with tempfile.TemporaryDirectory() as directory:
    random.seed(29); np.random.seed(29)
    diagnostic_rows, diagnostic_targets = saa_run(
        target.copy(), pool_count.clone(),
        considered_atts=ATTRIBUTES, ordered_to_adjust_atts=ATTRIBUTES,
        last_adjustment_order=ATTRIBUTES, max_run_time=3, extra_rm_frac=0.35,
        output_each_step=True, meta_output_dir=Path(directory),
        stochastic_mode='count_native_ilp',
    )
    adjustment_diagnostics = pd.read_csv(Path(directory) / 'direct_count_adjustments.csv')
    loopback_diagnostics = pd.read_csv(Path(directory) / 'direct_count_loopbacks.csv')
    assert len(loopback_diagnostics) == len(diagnostic_targets)
    assert set(adjustment_diagnostics['mode']) == {'count_native_ilp'}
    print(adjustment_diagnostics[[
        'pass', 'adjustment_index', 'active_attribute', 'ilp_cell_count',
        'stage1_status', 'stage2_status', 'marginal_l1_error',
        'ordinary_zero_cells_made_positive',
    ]])
    print(loopback_diagnostics)


   pass  adjustment_index active_attribute  ilp_cell_count stage1_status  \
0     0                 0           hhsize               0        NotRun   
1     0                 1         vehicles               9       Optimal   
2     0                 2           income              12       Optimal   
3     1                 0         vehicles               0        NotRun   
4     1                 1           hhsize               3       Optimal   
5     1                 2           income               6       Optimal   
6     2                 0           hhsize               0        NotRun   
7     2                 1         vehicles               3       Optimal   
8     2                 2           income               2       Optimal   

  stage2_status  marginal_l1_error  ordinary_zero_cells_made_positive  
0        NotRun                  0                                  0  
1       Optimal                  0                                  0  
2       Optimal        

## Interpretation

The three modes are comparable on population totals, controlled marginals, output schema, diagnostics, and distance from the source pool. Exact joint equality is intentionally not a requirement. The new mode performs no pool sampling or TRS inside an adjustment pass; only optional loop-back retention uses stochastic draws.